# ProteinBERT + PSSM1110 + CTD 融合模型

**目的**：在 ProteinBERT + PSSM1110 融合模型的基础上，额外拼接 AcrPred 论文中使用的 **CTD 168 维特征**（8 种理化性质的 Composition-Transition-Distribution），观察多模态特征融合是否能进一步提升性能。

**特征组成**：
- PSSM 1110 维（与 `fusion_best_metrics_for_acrpred_comparison.ipynb` 相同）
- CTD 168 维（从 AcrPred 论文的 `feature188d.py` 严格移植，8 种理化性质 × 21d）
- 合计 **1278 维**特征输入 PSSM 分支

**参考**：`fusion_best_metrics_for_acrpred_comparison.ipynb`（基线配置）；`02_AcrPred_reproduction.ipynb`（CTD 特征实现）

## 1. 依赖与路径

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from tensorflow import keras

from proteinbert import (
    load_anticrispr_with_ids,
    load_pretrained_model,
    FusionTrainConfig,
    load_feature_cache,
    attach_pssm_features,
)
from proteinbert.pssm_fusion import (
    _build_late_fusion_model,
    _encode_x,
    expected_calibration_error,
    find_best_threshold,
)

PROJECT_ROOT = '/home/nemophila/projects/protein_bert'
BENCHMARKS_DIR = f'{PROJECT_ROOT}/anticrispr_benchmarks'
WORK_ROOT = os.environ.get('PSSM_WORK_ROOT', '/home/nemophila/data/pssm_work')
FEAT_DIR = f'{WORK_ROOT}/features'

# 与 full experiment / confusion_matrix_demo 一致，固定种子便于复现与对比
SEED = 22
PSSM_VARIANT = '1110'

2026-03-09 17:47:19.830835: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


## 2. 扩展评估函数（含 ACC, SN, SP，与 AcrPred 论文指标对齐）

In [2]:
def evaluate_binary_full(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    """
    二分类完整指标，包含 AcrPred 论文中的 ACC/SN/SP 以及 AUPRC/F1/MCC/Brier/ECE。
    正类为 Acr (label=1)，SN = Sensitivity = Recall for positive, SP = Specificity = TN/(TN+FP)。
    """
    y_cls = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_cls).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    acc = accuracy_score(y_true, y_cls)
    return {
        'AUC': float(roc_auc_score(y_true, y_prob)),
        'AUPRC': float(average_precision_score(y_true, y_prob)),
        'F1': float(f1_score(y_true, y_cls)),
        'MCC': float(matthews_corrcoef(y_true, y_cls)),
        'Brier': float(brier_score_loss(y_true, y_prob)),
        'ECE': float(expected_calibration_error(y_true, y_prob, n_bins=10)),
        'ACC': float(acc),
        'SN': float(sn),
        'SP': float(sp),
        'Threshold': float(threshold),
    }

## 3. 加载 PSSM 数据 + 提取 CTD 168d 特征

In [3]:
train_base_df, test_base_df = load_anticrispr_with_ids(BENCHMARKS_DIR, benchmark_name='anticrispr_binary')
parquet_path = f'{FEAT_DIR}/pssm_features_{PSSM_VARIANT}.parquet'
csv_path = f'{FEAT_DIR}/pssm_features_{PSSM_VARIANT}.csv'
cache_path = parquet_path if os.path.exists(parquet_path) else csv_path
if not os.path.exists(cache_path):
    raise FileNotFoundError(f'PSSM cache not found: {cache_path}')

feature_df, pssm_feature_cols = load_feature_cache(cache_path)
train_df = attach_pssm_features(train_base_df, feature_df, pssm_feature_cols)
test_df = attach_pssm_features(test_base_df, feature_df, pssm_feature_cols)

# ===== CTD 168d 特征提取（从 AcrPred feature188d.py 严格移植） =====
PP = [
    [['R','K','E','D','Q','N'], ['G','A','S','T','P','H','Y'], ['C','L','V','I','M','F','W']],
    [['G','A','S','T','P','D','C'], ['N','V','E','Q','I','L'], ['M','H','K','F','R','Y','W']],
    [['L','I','F','W','C','M','V','Y'], ['P','A','T','G','S'], ['H','Q','R','K','N','E','D']],
    [['G','A','S','D','T'], ['C','P','N','V','E','Q','I','L'], ['K','M','H','F','R','Y','W']],
    [['K','R'], ['A','N','C','Q','G','H','I','L','M','F','P','S','T','W','Y','V'], ['D','E']],
    [['E','A','L','M','Q','K','R','H'], ['V','I','Y','C','W','F','T'], ['G','N','P','S','D']],
    [['A','L','F','C','G','I','V','W'], ['P','K','Q','E','N','D'], ['M','R','S','T','H','Y']],
    [['G','Q','D','N','A','H','R'], ['K','T','S','E','C'], ['I','L','M','F','P','W','Y','V']],
]

def ctd_168d(seq: str) -> np.ndarray:
    L = len(seq)
    ctd = []
    for j in range(8):
        n1 = np.zeros(3)
        n2 = np.zeros(3)
        n3 = [[], [], []]
        for k in range(L):
            for g in range(3):
                if seq[k] in PP[j][g]:
                    n1[g] += 1
                    n3[g].append(k + 1)
                    if k + 1 < L:
                        if g == 0:
                            if seq[k + 1] in PP[j][1]: n2[0] += 1
                            elif seq[k + 1] in PP[j][2]: n2[1] += 1
                        elif g == 1:
                            if seq[k + 1] in PP[j][0]: n2[0] += 1
                            elif seq[k + 1] in PP[j][2]: n2[2] += 1
                        elif g == 2:
                            if seq[k + 1] in PP[j][0]: n2[1] += 1
                            elif seq[k + 1] in PP[j][1]: n2[2] += 1
                    break
        c = n1 / L
        t = n2 / max(L - 1, 1)
        d = np.zeros((3, 5))
        for g in range(3):
            cnt = int(n1[g])
            if cnt > 0: d[g][0] = n3[g][0] / L
            if cnt >= 4: d[g][1] = n3[g][int(0.25 * cnt) - 1] / L
            if cnt >= 2: d[g][2] = n3[g][int(0.5 * cnt) - 1] / L
            if cnt >= 2: d[g][3] = n3[g][int(0.75 * cnt) - 1] / L
            if cnt > 0: d[g][4] = n3[g][cnt - 1] / L
        ctd.extend(c.tolist())
        ctd.extend(t.tolist())
        ctd.extend(d.flatten().tolist())
    return np.array(ctd, dtype=np.float64)

# 为 train 和 test 计算 CTD 特征并拼接为 DataFrame 列
ctd_col_names = [f'ctd_{i}' for i in range(168)]

for df in [train_df, test_df]:
    ctd_matrix = np.array([ctd_168d(s) for s in df['seq']])
    for i, col in enumerate(ctd_col_names):
        df[col] = ctd_matrix[:, i]

# 合并特征列: PSSM (1110d) + CTD (168d) = 1278d
feature_cols = list(pssm_feature_cols) + ctd_col_names

y_test = test_df['label'].astype(int).to_numpy()
print(f'train: {train_df.shape}, test: {test_df.shape}')
print(f'Feature dims: PSSM={len(pssm_feature_cols)}, CTD=168, Total={len(feature_cols)}')

/tmp/ipykernel_58240/2337943766.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = ctd_matrix[:, i]


train: (1107, 1281), test: (286, 1281)
Feature dims: PSSM=1110, CTD=168, Total=1278


## 4. 训练 ProteinBERT + PSSM1110 + CTD 融合模型

In [4]:
cfg = FusionTrainConfig(
    seq_len=512,
    batch_size=8,
    frozen_epochs=6,
    unfrozen_epochs=12,
    frozen_lr=1e-4,
    unfrozen_lr=2e-5,
    pssm_dropout=0.3,
    global_dropout=0.3,
    pssm_hidden_dim=128,
    global_hidden_dim=128,
    global_bottleneck_dim=64,
    fusion_hidden_dim=128,
    use_hidden_global_concat=True,
)

rng_train, rng_valid = train_test_split(
    train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED
)

x_train = rng_train[feature_cols].to_numpy(dtype=np.float32)
x_valid = rng_valid[feature_cols].to_numpy(dtype=np.float32)
x_test = test_df[feature_cols].to_numpy(dtype=np.float32)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_valid = scaler.transform(x_valid)
x_test = scaler.transform(x_test)

y_train = rng_train['label'].astype(int).to_numpy()
y_valid = rng_valid['label'].astype(int).to_numpy()

pmg, enc = load_pretrained_model(
    local_model_dump_dir=f'{PROJECT_ROOT}/proteinbert_models',
    download_model_dump_if_not_exists=True,
    validate_downloading=False,
)

X_train = _encode_x(enc, rng_train['seq'].tolist(), cfg.seq_len, x_train)
X_valid = _encode_x(enc, rng_valid['seq'].tolist(), cfg.seq_len, x_valid)
X_test = _encode_x(enc, test_df['seq'].tolist(), cfg.seq_len, x_test)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=cfg.patience, restore_best_weights=True
    )
]

model = _build_late_fusion_model(
    pmg, seq_len=cfg.seq_len, pssm_dim=len(feature_cols),
    freeze_pretrained_layers=True, cfg=cfg,
)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.frozen_lr), loss='binary_crossentropy')
model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
          epochs=cfg.frozen_epochs, batch_size=cfg.batch_size, callbacks=callbacks, verbose=1)

for layer in model.layers:
    layer.trainable = True
model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg.unfrozen_lr), loss='binary_crossentropy')
model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
          epochs=cfg.unfrozen_epochs, batch_size=cfg.batch_size, callbacks=callbacks, verbose=1)

valid_prob = model.predict(X_valid, batch_size=cfg.batch_size, verbose=0).reshape(-1)
thr = find_best_threshold(y_valid, valid_prob)
test_prob = model.predict(X_test, batch_size=cfg.batch_size, verbose=0).reshape(-1)

print('Best threshold (valid F1):', thr)

2026-03-09 17:47:22.847387: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2026-03-09 17:47:22.848104: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2026-03-09 17:47:22.852799: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:2a:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-03-09 17:47:22.852926: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 1 with properties: 
pciBusID: 0000:ab:00.0 name: NVIDIA L40S computeCapability: 8.9
coreClock: 2.52GHz coreCount: 142 deviceMemorySize: 44.53GiB deviceMemoryBandwidth: 804.75GiB/s
2026-03-09 17:47:22.852940: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2026-03-09 17:47:22.8

Epoch 1/6


2026-03-09 17:47:31.499005: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.11
2026-03-09 17:47:32.347561: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublasLt.so.11
2026-03-09 17:47:32.361051: I tensorflow/stream_executor/cuda/cuda_blas.cc:1838] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-03-09 17:47:32.362513: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudnn.so.8
2026-03-09 17:47:34.858567: W tensorflow/stream_executor/gpu/asm_compiler.cc:63] Running ptxas --version returned 256
2026-03-09 17:47:35.002915: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] Internal: ptxas exited with non-zero error code 256, output: 
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.


125/125 [==============================] - 32s 142ms/step - loss: 0.4906 - val_loss: 0.3319
Epoch 2/6
125/125 [==============================] - 4s 31ms/step - loss: 0.3584 - val_loss: 0.2569
Epoch 3/6
125/125 [==============================] - 4s 31ms/step - loss: 0.2708 - val_loss: 0.2226
Epoch 4/6
125/125 [==============================] - 4s 31ms/step - loss: 0.2534 - val_loss: 0.2153
Epoch 5/6
125/125 [==============================] - 4s 31ms/step - loss: 0.2034 - val_loss: 0.1874
Epoch 6/6
125/125 [==============================] - 4s 31ms/step - loss: 0.1343 - val_loss: 0.1965
Epoch 1/12
125/125 [==============================] - 33s 140ms/step - loss: 0.1381 - val_loss: 0.2023
Epoch 2/12
125/125 [==============================] - 7s 55ms/step - loss: 0.1089 - val_loss: 0.1983
Epoch 3/12
125/125 [==============================] - 7s 54ms/step - loss: 0.0892 - val_loss: 0.1918
Epoch 4/12
125/125 [==============================] - 7s 55ms/step - loss: 0.0609 - val_loss: 0.1891
Ep

## 5. 完整指标与混淆矩阵

In [5]:
metrics = evaluate_binary_full(y_test, test_prob, threshold=thr)
for k, v in metrics.items():
    print(f'{k}: {v}')

y_pred = (test_prob >= thr).astype(int)
cm = confusion_matrix(y_test, y_pred)
print('\nConfusion matrix (test):')
print(cm)

AUC: 0.9307692307692308
AUPRC: 0.7225048161856057
F1: 0.68
MCC: 0.6500269127331219
Brier: 0.045590249419583645
ECE: 0.03781599802831253
ACC: 0.9440559440559441
SN: 0.6538461538461539
SP: 0.9730769230769231
Threshold: 0.39999999999999997

Confusion matrix (test):
[[253   7]
 [  9  17]]


## 6. 与基线对比

对比 **ProteinBERT + PSSM1110 + CTD** 融合模型 vs 原始 PSSM1110-only 融合模型 及 AcrPred 论文结果。

In [6]:
acrpred_paper = {
    'AUC': 0.952, 'ACC': 0.881, 'SN': 0.923, 'SP': 0.877,
    'AUPRC': None, 'F1': None, 'MCC': None, 'Brier': None, 'ECE': None,
}

import json
ours_base_path = '/home/nemophila/projects/protein_bert/Comparison/results/ours_fusion_metrics.json'
with open(ours_base_path) as f:
    ours_base = json.load(f)['metrics']

ours_ctd = {k: round(metrics[k], 4) for k in ['AUC','ACC','SN','SP','AUPRC','F1','MCC','Brier','ECE']}

comparison = pd.DataFrame({
    'AcrPred (paper)': [acrpred_paper.get(k, '—') if acrpred_paper.get(k) is not None else '—' for k in ours_ctd.keys()],
    'Ours (PSSM1110)': [round(ours_base.get(k, 0), 4) for k in ours_ctd.keys()],
    'Ours (PSSM1110+CTD)': list(ours_ctd.values()),
}, index=list(ours_ctd.keys()))
comparison


,AcrPred (paper),Ours (PSSM1110),Ours (PSSM1110+CTD)
AUC,0.952,0.9355,0.9308
ACC,0.881,0.9091,0.9441
SN,0.923,0.8077,0.6538
SP,0.877,0.9192,0.9731
AUPRC,—,0.7071,0.7225
F1,—,0.6176,0.6800
MCC,—,0.5904,0.6500
Brier,—,0.0559,0.0456
ECE,—,0.0547,0.0378
